# 01 - Dataset (2019-2026): per-cell 50 km GOES/GLM signature features

The single cache builder for the whole project. Task: predict which 50 km CONUS-land cells
flood on CDT day D from the previous day's (D-1) GOES/GLM. **Everything is on the 50 km grid -
no full-resolution image.** Deep models convolve over the 59x95 feature grid; tabular models
(XGB / LR) use the same features per cell. **Pure satellite signatures - no climatology.**

## What gets cached, per sample (one CDT day)

1. **seq** - per-cell **sequence** features, **(8, 19, 59, 95)** f32, masked-pooled from the
   FULL-res pixels each 3h frame: raw cell-mean BT (b8, b10, b11, b14, b15); BT differences
   (10-8, 11-14, 14-15, 14-10); cold-cloud intensity (b14_min, b14_p05, frac<235, frac<220);
   temporal (dt_b14_3h, running max cooling, cold-area change<220); GLM (flash count, density,
   occurrence).
2. **sum** - per-cell **daily summaries**, **(8, 59, 95)** f32, over the 8 frames: min_b14_24h,
   min_b8_24h, max_frac<220, hours_with_frac<235, max_cooling_24h, daily_flash_count,
   max_3h_flash_count, hours_with_lightning.
3. **t** - per-frame lead time (8,), hours before day D / 24.
4. **y** - the label (59, 95), a 0/1 observed-flood map (Groundsource union NCEI storm events).

Bands read: **B8, B10, B11, B14, B15** (the ABI+GLM precipitation IR set). All emissive, so
readable day and night. Per-sample on disk ~= 3.6 MB (whole cache ~9 GB).

## Split

The manifest marks only the CV pool: **2019-2025 -> "cv"**, **2026 -> "unused"** (truncated
labels). The train/val/test split is **not** stored here - it is blocked K-fold
cross-validation computed at train time by `floodlens/foldsplit.py` (one fixed test
set + rotating train/val folds, each spanning all years). Days are CDT (UTC-5); GOES stays
UTC (one CDT day spans two UTC folders). Storm events shift -5 h to CDT; Groundsource dates
day-resolution, used as-is.

## 0. Config and imports

Feature definitions and thresholds live here; the grid, the year span (YEARS), and the
cache location (CACHE_DIR) come from the repo-root config.py — the single source of truth
the trainers also read. The cache is `cache/goes_grid50_2019_2026` on the root NVMe.

In [ ]:
import json
import sys
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd

from floodlens.config import (CACHE_DIR, CELL_KM, DATA_DIR, GLM_DIR, IMG_H, IMG_W,
                              T_FRAMES, YEARS, UNIFIED_PARQUET, build_grid_cells,
                              grid_transform)
from floodlens.gridindex import build_pix2cell

# --- full dataset 2019-2026; pure per-cell (50 km) feature cache ---

# The manifest carries only a coarse marker: 2026 is "unused" (its groundsource labels stop
# 2026-01-28 and storm events lag, so its 0.38% base rate is a truncated-label artifact),
# and 2019-2025 is the CV pool ("cv"). The actual train/val/test split is blocked K-fold
# cross-validation computed at train time by floodlens/foldsplit.py.
LABELS_END_YEAR = 2025

# bands read per NetCDF (ABI+GLM precip IR set). Everything is pooled to per-cell 50 km
# features -- NO full-resolution image is stored: the deep models convolve over the 59x95
# feature grid, and the tabular models use the same features per cell.
RAW_BANDS = (8, 10, 11, 14, 15)        # B8/10/11/14/15

# cold-cloud / deep-convection thresholds (Kelvin)
CC235, CC220 = 235.0, 220.0            # b14 cold-cloud-top fractions (convective coverage/depth)

print(f"years {min(YEARS)}-{max(YEARS)}  |  bands {RAW_BANDS} -> per-cell 50 km features")
print(f"split: CV pool 2019-{LABELS_END_YEAR} (blocked K-fold at train time); 2026 dropped")
print(f"cache -> {CACHE_DIR}")

## 1. Output grid and the masked-pooling index

The label space and the per-cell features both live on the same fixed grid of 50 km CONUS
cells (config.build_grid_cells, EPSG:5070). The pixel→cell map (gridindex.build_pix2cell)
assigns every full-resolution ABI pixel to its cell; we turn it into a sorted index so
each cell's pixels form a contiguous block. That gives true masked pooling — every
aggregation (mean, min, fraction) is computed over exactly the valid pixels that fall in a
cell, never an image resize.

In [ ]:
cells, GRID_R, GRID_C, land_mask = build_grid_cells()
p2c, _, _, _ = build_pix2cell()                       # (1500,2500) -> flat cell id, -1 off-grid
N_CELL = GRID_R * GRID_C
n_land = int(land_mask.sum())

# group valid pixels by cell: VIDX gathers pixel values into cell-contiguous blocks
_flat = p2c.ravel()
_vidx = np.flatnonzero(_flat >= 0)
_cell = _flat[_vidx].astype(np.int64)
_order = np.argsort(_cell, kind="stable")
VIDX = _vidx[_order]                                   # pixel linear indices, grouped by cell
CELLG = _cell[_order]                                  # cell id per gathered pixel (ascending)
print(f"grid {GRID_R}x{GRID_C}  |  land cells {n_land}  |  pooled pixels {len(VIDX):,}")


def pool_means(stack):
    """(C,H,W) physical -> (C,GRID_R,GRID_C): masked mean of each channel over a cell's pixels."""
    C = stack.shape[0]
    v = stack.reshape(C, -1)[:, VIDX]
    fin = np.isfinite(v)
    v0 = np.where(fin, v, 0.0)
    out = np.full((C, N_CELL), np.nan, np.float32)
    for c in range(C):
        s = np.bincount(CELLG, weights=v0[c], minlength=N_CELL)
        n = np.bincount(CELLG, weights=fin[c].astype(np.float64), minlength=N_CELL)
        nz = n > 0
        out[c, nz] = (s[nz] / n[nz]).astype(np.float32)
    return out.reshape(C, GRID_R, GRID_C)


def pool_frac(a, thr):
    """Fraction of a cell's valid pixels with value < thr."""
    v = a.ravel()[VIDX]
    fin = np.isfinite(v)
    below = np.where(fin, (v < thr), 0.0).astype(np.float64)
    s = np.bincount(CELLG, weights=below, minlength=N_CELL)
    n = np.bincount(CELLG, weights=fin.astype(np.float64), minlength=N_CELL)
    out = np.zeros(N_CELL, np.float32)
    nz = n > 0
    out[nz] = (s[nz] / n[nz]).astype(np.float32)
    return out.reshape(GRID_R, GRID_C)


def pool_min(a):
    """Per-cell minimum of a channel (masked)."""
    v = a.ravel()[VIDX].astype(np.float64)
    v = np.where(np.isfinite(v), v, np.inf)
    out = np.full(N_CELL, np.inf)
    np.minimum.at(out, CELLG, v)
    out[~np.isfinite(out)] = np.nan
    return out.reshape(GRID_R, GRID_C).astype(np.float32)


def pool_pctl(a, q):
    """Per-cell q-th percentile of a channel (masked). Loops cells; land-only is cheap."""
    v = a.ravel()[VIDX].astype(np.float32)
    out = np.full(N_CELL, np.nan, np.float32)
    edges = np.searchsorted(CELLG, np.arange(N_CELL + 1))
    for c in range(N_CELL):
        s, e = edges[c], edges[c + 1]
        if e > s:
            seg = v[s:e]
            seg = seg[np.isfinite(seg)]
            if seg.size:
                out[c] = np.percentile(seg, q)
    return out.reshape(GRID_R, GRID_C)

## 2. Labels — observed floods to daily 0/1 cell maps

Over all of 2019-2026. A cell is positive on CDT day D if a Groundsource news-report extent
or an NCEI storm-event footprint intersects it. Storm events shift −5 h to CDT; Groundsource
dates (day-resolution) are used as-is. An event spanning up to 5 CDT days marks each day;
longer collapses to its issue day.

In [ ]:
LABEL_SOURCES = {"groundsource", "storm_event"}
CDT = pd.Timedelta(hours=5)
MAX_SPLIT_DAYS = 5

u = gpd.read_parquet(UNIFIED_PARQUET)
w = u[u["issue_date"].dt.year.isin(YEARS) & u["source"].isin(LABEL_SOURCES)].copy()

shift = pd.Series(pd.Timedelta(0), index=w.index)
shift[w["source"] == "storm_event"] = CDT             # only storm events carry a real UTC time
w["issue_cdt"] = w["issue_date"] - shift
w["expire_cdt"] = w["expire_date"] - shift


def event_days(issue, expire):
    "CDT days an event labels: each day if it spans <= MAX_SPLIT_DAYS, else the issue day."
    days = pd.date_range(issue.normalize(), expire.normalize(), freq="D")
    return days if len(days) <= MAX_SPLIT_DAYS else days[:1]


w["days"] = [event_days(i, e) for i, e in zip(w["issue_cdt"], w["expire_cdt"])]
ev = w.explode("days", ignore_index=True).rename(columns={"days": "label_day"})
ev = ev[ev["label_day"].dt.year.isin(YEARS)]

j = gpd.sjoin(cells, ev[["label_day", "geometry"]], predicate="intersects")
labels = {}
for day, g in j.groupby("label_day"):
    a = np.zeros((GRID_R, GRID_C), np.float32)
    a[g["R"], g["C"]] = 1.0
    labels[day.date()] = a

cnt = w["source"].value_counts()
pos_per_day = np.array([a[land_mask].sum() for a in labels.values()])
print(f"{len(w):,} flood events ({' + '.join(f'{v:,} {k}' for k, v in cnt.items())}) "
      f"-> {len(ev):,} event-days on {len(labels)} CDT days, {min(YEARS)}-{max(YEARS)}")
print(f"flooded land cells/day: median {np.median(pos_per_day):.0f}, max {pos_per_day.max():.0f} "
      f"({np.median(pos_per_day)/n_land:.1%} of land)")
# label days per calendar year
_ly = pd.Series([d.year for d in labels]).value_counts().sort_index()
print("label days/year:", {int(y): int(c) for y, c in _ly.items()})

## 3. GOES file and time helpers

One NetCDF per scan on /mnt/disk4, 8 frames per UTC day (00, 03, …, 21 UTC). goes_cdt_day
returns the 8 frames whose CDT date (scan UTC − 5 h) equals the requested day, globbing the
two UTC folders the CDT day spans.

In [ ]:
def _scan_token(p):
    "The s-token (s{YYYYDDDHHMM...}) from a GOES filename."
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _scan_dt(p):
    "Scan-start datetime (UTC) parsed from the filename token."
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


def goes_files(d):
    "Sorted GOES NetCDFs in the UTC day folder d (GOES16/19 auto-globbed)."
    pat = f"*/{d.year}/{d.month:02d}/{d.day:02d}/*.nc"
    return sorted(DATA_DIR.glob(pat), key=_scan_token)


def goes_cdt_day(cdt_day):
    "GOES frames whose CDT date (scan UTC - 5 h) equals cdt_day, in scan order."
    nxt = cdt_day + timedelta(days=1)
    cand = sorted(goes_files(cdt_day) + goes_files(nxt), key=_scan_token)
    return [f for f in cand if (_scan_dt(f) - timedelta(hours=5)).date() == cdt_day]


def valid_day(label_day):
    "True if CDT day D-1 has all T_FRAMES GOES frames."
    return len(goes_cdt_day(label_day - timedelta(days=1))) == T_FRAMES


_valid = []                                            # first few valid days (short-circuit)
for d in sorted(labels):
    if valid_day(d):
        _valid.append(d)
        if len(_valid) >= 6:
            break
example_label_day = _valid[5]
example_in_day = example_label_day - timedelta(days=1)
_files = goes_cdt_day(example_in_day)
print(f"CDT D-1 input {example_in_day}  ->  flood day {example_label_day}")
print(f"{len(_files)} frames; scan UTC:", [f"{_scan_dt(f):%H:%M}" for f in _files])

## 4. Read the raw ABI bands

For one CDT day (D-1), read the raw ABI bands (b8, b10, b11, b14, b15) across the 8 frames -
full-res physical (Kelvin). These pixels are pooled into per-cell 50 km features below; no
full-resolution image is stored.

In [ ]:
B_INDEX = {b: i for i, b in enumerate(RAW_BANDS)}


def read_bands(f):
    "Read RAW_BANDS from one GOES NetCDF -> (len(RAW_BANDS),H,W) physical (K), space NaN."
    out = np.empty((len(RAW_BANDS), IMG_H, IMG_W), np.float32)
    with netCDF4.Dataset(f) as nc:
        for i, b in enumerate(RAW_BANDS):
            out[i] = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
    return out


def read_raw_stack(in_day):
    "(T,nbands,H,W) physical raw bands for CDT day in_day, plus files and scan times."
    files = goes_cdt_day(in_day)[:T_FRAMES]
    raw = np.stack([read_bands(f) for f in files])        # (T,len(RAW_BANDS),H,W)
    return raw, files, [_scan_dt(f) for f in files]


_raw, _f, _times = read_raw_stack(example_in_day)
print(f"raw bands {_raw.shape} {_raw.dtype} (full-res physical; pooled to cells below)")
for i, b in enumerate(RAW_BANDS):
    a = _raw[:, i]
    print(f"  b{b:>2}: min {np.nanmin(a):8.2f}  max {np.nanmax(a):8.2f}")

## 5. Per-cell sequence features (masked pooled)

Pool the raw bands onto the 50 km grid, **per 3h frame**, into 19 per-cell signature features
(`seq_cell_features`): raw cell-mean BT (b8, b10, b11, b14, b15), the four BT differences
(10-8, 11-14, 14-15, 14-10), cold-cloud intensity (b14_min, b14_p05, frac<235, frac<220),
temporal evolution (3-hourly d-b14, running max cooling, cold-area change<220), and GLM
(flash count / density / occurrence, filled in from the GLM cell below). Result: (8, 19, R, C).
GLM channels are set here as placeholders and populated once GLM counts are computed.

In [ ]:
SEQ_NAMES = [
    "b8", "b10", "b11", "b14", "b15",                          # raw cell-mean BT (5)
    "btd_10-8", "btd_11-14", "btd_14-15", "btd_14-10",         # BT differences (4)
    "b14_min", "b14_p05", "frac_b14<235", "frac_b14<220",      # cold-cloud intensity (4)
    "dt_b14_3h", "max_cool_b14", "cold_area_chg_220",          # temporal evolution (3)
    "glm_count", "glm_density", "glm_occurrence",              # GLM (3)
]
N_SEQ = len(SEQ_NAMES)                    # 19 per-cell features per 3h frame
CELL_AREA_KM2 = float(CELL_KM) ** 2       # for GLM flash density


def seq_cell_features(raw, glm_count):
    "(T,nb,H,W) raw + (T,R,C) GLM counts -> (T,19,R,C) per-frame per-cell features."
    T = raw.shape[0]
    out = np.zeros((T, N_SEQ, GRID_R, GRID_C), np.float32)
    prev_b14, prev_f220 = None, None
    maxcool = np.zeros((GRID_R, GRID_C), np.float32)
    for t in range(T):
        b8 = raw[t, B_INDEX[8]]
        b10 = raw[t, B_INDEX[10]]
        b11 = raw[t, B_INDEX[11]]
        b14 = raw[t, B_INDEX[14]]
        b15 = raw[t, B_INDEX[15]]
        b14m = pool_means(b14[None])[0]
        out[t, 0] = pool_means(b8[None])[0]                    # raw cell-mean BT
        out[t, 1] = pool_means(b10[None])[0]
        out[t, 2] = pool_means(b11[None])[0]
        out[t, 3] = b14m
        out[t, 4] = pool_means(b15[None])[0]
        out[t, 5] = pool_means((b10 - b8)[None])[0]            # BTDs
        out[t, 6] = pool_means((b11 - b14)[None])[0]
        out[t, 7] = pool_means((b14 - b15)[None])[0]
        out[t, 8] = pool_means((b14 - b10)[None])[0]
        out[t, 9] = pool_min(b14)                              # coldest pixel
        out[t, 10] = pool_pctl(b14, 5)                         # 5th-pctl (robust cold)
        f235 = pool_frac(b14, CC235)
        f220 = pool_frac(b14, CC220)
        out[t, 11] = f235
        out[t, 12] = f220
        dt = 0.0 if prev_b14 is None else np.nan_to_num(b14m - prev_b14)   # +ve = warming
        out[t, 13] = dt
        maxcool = np.maximum(maxcool, np.maximum(-dt, 0.0))    # running max cooling
        out[t, 14] = maxcool
        out[t, 15] = 0.0 if prev_f220 is None else np.nan_to_num(f220 - prev_f220)
        prev_b14, prev_f220 = b14m, f220
    out[:, 16] = glm_count                                     # flash count
    out[:, 17] = glm_count / CELL_AREA_KM2                     # flash density (per km^2)
    out[:, 18] = (glm_count > 0).astype(np.float32)            # lightning occurrence
    return out


print(f"defined seq_cell_features -> {N_SEQ} per-frame features:")
print("  ", SEQ_NAMES)

## 6. GLM lightning flash counts

For each sample, load the GLM flash parquet(s) for the UTC dates the 8 frames span, assign
each flash to its nearest 3-hour frame and its 50 km cell (Albers integer binning), and count
flashes per cell per frame. Returns (8, R, C) counts; the sequence builder derives flash
**count**, **density** (per km^2), and **occurrence** (any flash) from it, and the daily
summary derives daily count, max-3h count, and hours-with-lightning.

In [ ]:
import pyproj

GX0, GY0, GSTEP, GGR, GGC = grid_transform()
assert (GGR, GGC) == (GRID_R, GRID_C), "grid_transform disagrees with build_grid_cells"
_ALBERS = pyproj.Transformer.from_crs(4326, 5070, always_xy=True)
GLM_COLS = ["time_start", "lat", "lon"]


def _flash_rc(lon, lat):
    "Map flash lon/lat -> (R, C, in_grid) on the 50 km grid via Albers integer binning."
    x, y = _ALBERS.transform(np.asarray(lon), np.asarray(lat))
    x, y = np.asarray(x), np.asarray(y)
    C = np.floor((x - GX0) / GSTEP)
    R = (GGR - 1) - np.floor((y - GY0) / GSTEP)
    ok = np.isfinite(x) & np.isfinite(y) & (C >= 0) & (C < GGC) & (R >= 0) & (R < GGR)
    return (np.where(ok, R, 0).astype(np.int64), np.where(ok, C, 0).astype(np.int64), ok)


def _load_glm(dates):
    dfs = []
    for d in dates:
        pth = GLM_DIR / f"{d.year}" / f"glm_flashes_{d:%Y%m%d}.parquet"
        if pth.exists():
            dfs.append(pd.read_parquet(pth, columns=GLM_COLS))
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)


def glm_cell_features(frame_times):
    "(T,R,C) per-3h-bin GLM flash counts on the 50 km grid (count -> density/occurrence)."
    out = np.zeros((T_FRAMES, GRID_R, GRID_C), np.float32)
    g = _load_glm(sorted({pd.Timestamp(ft).date() for ft in frame_times}))
    if g is None or len(g) == 0:
        return out
    t = g["time_start"].to_numpy().astype("datetime64[s]")
    ft = np.array([np.datetime64(pd.Timestamp(x), "s") for x in frame_times])
    dmat = np.abs((t[:, None] - ft[None, :]).astype("timedelta64[s]").astype(np.float64))
    bin_idx = dmat.argmin(1)
    keep = dmat.min(1) <= 5400.0                            # within 1.5 h of a frame
    R, C, ok = _flash_rc(g["lon"].to_numpy(), g["lat"].to_numpy())
    keep &= ok
    bin_idx, R, C = bin_idx[keep], R[keep], C[keep]
    cell = R * GRID_C + C
    for b in range(T_FRAMES):
        m = bin_idx == b
        if not m.any():
            continue
        cnt = np.bincount(cell[m], minlength=N_CELL).astype(np.float64)
        out[b] = cnt.reshape(GRID_R, GRID_C).astype(np.float32)
    return out


_glm = glm_cell_features(_times)
print(f"GLM flash-count grids {_glm.shape}  {_glm.dtype}")
print(f"  example day total flashes on land: {_glm[:, land_mask].sum():.0f}")

## 7. Daily summaries

Collapse the 8 frames to one map per cell (`daily_summary`, (8, R, C)): min_b14_24h,
min_b8_24h, max_frac_b14<220, hours_with_frac_b14<235, max_cooling_b14_24h,
daily_flash_count, max_3h_flash_count, and hours_with_lightning - the day-level intensity /
persistence signals distilled from the per-frame sequence.

In [ ]:
SUM_NAMES = [
    "min_b14_24h", "min_b8_24h", "max_frac_b14<220", "hours_frac_b14<235",
    "max_cool_b14_24h", "glm_daily_count", "glm_max_3h", "glm_hours_lit",
]
N_SUM = len(SUM_NAMES)                     # 8 daily summaries per cell


def daily_summary(seq):
    "(T,19,R,C) sequence -> (8,R,C) daily summaries over the 8 D-1 frames."
    b14min = seq[:, 9]                                      # per-frame coldest pixel
    b8mean = seq[:, 0]
    f235, f220 = seq[:, 11], seq[:, 12]
    maxcool = seq[:, 14]                                    # running max cooling
    cnt = seq[:, 16]                                        # flash count
    out = np.zeros((N_SUM, GRID_R, GRID_C), np.float32)
    with warnings.catch_warnings():                         # ocean cells are all-NaN
        warnings.simplefilter("ignore", RuntimeWarning)
        out[0] = np.nanmin(b14min, axis=0)                  # min_b14_24h
        out[1] = np.nanmin(b8mean, axis=0)                  # min_b8_24h
    out[2] = np.nan_to_num(f220).max(0)                     # max_frac_b14<220
    out[3] = (np.nan_to_num(f235) > 0).sum(0) * 3.0         # hours_with_frac_b14<235
    out[4] = np.nan_to_num(maxcool).max(0)                  # max_cooling_b14_24h
    out[5] = np.nan_to_num(cnt).sum(0)                      # daily_flash_count
    out[6] = np.nan_to_num(cnt).max(0)                      # max_3h_flash_count
    out[7] = (np.nan_to_num(cnt) > 0).sum(0) * 3.0          # hours_with_lightning
    return np.nan_to_num(out)


_seq = seq_cell_features(_raw, _glm)
_sum = daily_summary(_seq)
print(f"sequence {_seq.shape} -> daily summary {_sum.shape}")
print(f"  {N_SUM} summaries:", SUM_NAMES)

## 8. (No image standardization)

There is no image branch, so no per-channel image standardization. The per-cell features are
kept in physical units (K, fractions, counts); each trainer normalizes them as it prefers.

In [ ]:
# No image standardization: model inputs are the per-cell feature grids, kept in physical
# units (K, fractions, counts). Each trainer normalizes them as it prefers (e.g. BatchNorm).
print("per-cell features kept in physical units - no image standardization needed")

## 9. Assemble one sample

build_sample ties it together: the standardized image stack (float16), the three per-cell
feature grids (float32), the normalized lead time, and the label.

In [ ]:
def lead_times(times, label_day):
    "Per-frame whole hours before day D's CDT start (D 05:00 UTC)."
    d05 = datetime(label_day.year, label_day.month, label_day.day, 5)
    return np.array([round((d05 - tt).total_seconds() / 3600) for tt in times], np.float32)


def build_sample(in_day, label_day):
    raw, files, times = read_raw_stack(in_day)            # (T,nb,H,W) full-res physical
    glm = glm_cell_features(times)                         # (T,R,C) flash counts
    seq = seq_cell_features(raw, glm)                      # (T,19,R,C) pooled from full-res
    summ = daily_summary(seq)                             # (8,R,C)
    lead = lead_times(times, label_day)
    return {"seq": seq.astype(np.float32),
            "sum": summ.astype(np.float32),
            "t": (lead / 24.0).astype(np.float32),        # normalized lead (hours/24)
            "y": labels[label_day].astype(np.uint8)}


t0 = time.perf_counter()
_s = build_sample(example_in_day, example_label_day)
print(f"one sample in {time.perf_counter()-t0:.1f}s:")
for k, v in _s.items():
    print(f"  {k:>4}: {tuple(v.shape)}  {v.dtype}  = {v.nbytes/1e6:.3f} MB")
print(f"  per-sample on disk ~= {sum(v.nbytes for v in _s.values())/1e6:.2f} MB")

## 10. Sample index

A sample is valid when all 8 GOES frames exist for CDT day D−1. We index every valid
(D−1 → D) pair and mark the CV pool: 2019-2025 → "cv", 2026 → "unused" (truncated labels,
excluded). The train/val/test split is not stored here — it is blocked K-fold
cross-validation computed at train time by `floodlens/foldsplit.py` (one fixed test
set + rotating train/val folds, each spanning all years).

In [ ]:
rows = []
for d_label in sorted(labels):                            # ~2.7k goes_cdt_day globs (HDD)
    d_in = d_label - timedelta(days=1)
    if len(goes_cdt_day(d_in)) == T_FRAMES:
        rows.append({"in_day": pd.Timestamp(d_in), "label_day": pd.Timestamp(d_label),
                     "n_pos": int(labels[d_label].sum())})
index = pd.DataFrame(rows)

yr = index["label_day"].dt.year
index["split"] = "cv"                                     # 2019-2025: the CV pool
index.loc[yr > LABELS_END_YEAR, "split"] = "unused"       # 2026: truncated labels, excluded

print(index["split"].value_counts().reindex(["cv", "unused"]).to_string())
print(f"\ntotal valid samples {min(YEARS)}-{max(YEARS)}: {len(index)}")
print("\nsamples per year x split:")
print(index.groupby([yr.rename("year"), "split"]).size().unstack(fill_value=0).to_string())
index.head()

## 11. Inspect one random sample - EVERY cached feature, all 8 frames

Everything is on the 50 km grid. Each per-frame `seq` feature is shown across all 8 D-1 frames
(rows = features, columns = frames), grouped: (1) raw ABI bands, (2) BT differences,
(3) cold-cloud intensity, (4) temporal evolution, (5) GLM; then (6) the 8 daily `sum` maps and
(7) the flood label. Together these are exactly what is cached.

In [ ]:
summer = index[index["label_day"].dt.month.isin(range(5, 10))]
r = (summer if len(summer) else index).sample(1, random_state=7).iloc[0]
ind, ld = r["in_day"].date(), r["label_day"].date()
samp = build_sample(ind, ld)
seq, summ, y, lead = samp["seq"], samp["sum"], samp["y"], samp["t"]
print(f"sample: input CDT D-1 {ind}  ->  flood day {ld}")
print(f"  flooded land cells: {int(y[land_mask].sum())}  |  total GLM flashes: "
      f"{seq[:, 16].sum():.0f}  |  per-frame lead (h): {(lead*24).astype(int).tolist()}")


def _mland(a):
    return np.where(land_mask, a, np.nan)


def _rows(idx, names, title, cmap):
    "Rows = SEQ features (by index), columns = the 8 D-1 frames (shared scale per row)."
    nf = len(names)
    fig, axes = plt.subplots(nf, T_FRAMES, figsize=(1.5 * T_FRAMES, 1.55 * nf),
                             squeeze=False)
    for ri, fi in enumerate(idx):
        row = _mland(seq[:, fi])
        vmin, vmax = np.nanmin(row), np.nanmax(row)
        for t in range(T_FRAMES):
            ax = axes[ri, t]
            ax.imshow(row[t], cmap=cmap, vmin=vmin, vmax=vmax)
            if ri == 0:
                ax.set_title(f"-{int(lead[t]*24)}h", fontsize=7)
            if t == 0:
                ax.set_ylabel(names[ri], fontsize=8, rotation=0, ha="right", va="center")
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def _panels(arrs, names, title, cmap="viridis", cols=4):
    n = len(names)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.1 * rows), squeeze=False)
    for k, ax in enumerate(axes.flat):
        if k < n:
            im = ax.imshow(_mland(arrs[k]), cmap=cmap)
            ax.set_title(names[k], fontsize=8)
            fig.colorbar(im, ax=ax, shrink=0.6)
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


# every cached per-cell feature, each across the 8 frames, in order (all on the 50 km grid)
_rows(range(0, 5),   SEQ_NAMES[0:5],  "1. Raw ABI bands (per-cell mean) - 8 frames", "gray_r")
_rows(range(5, 9),   SEQ_NAMES[5:9],  "2. BT differences - 8 frames", "RdBu_r")
_rows(range(9, 13),  SEQ_NAMES[9:13], "3. Cold-cloud intensity - 8 frames", "viridis")
_rows(range(13, 16), SEQ_NAMES[13:16], "4. Temporal evolution - 8 frames", "coolwarm")
_rows(range(16, 19), SEQ_NAMES[16:19], "5. GLM lightning - 8 frames", "hot")
_panels([summ[k] for k in range(N_SUM)], SUM_NAMES, "6. Daily summary features (8)")
_panels([np.where(y > 0, 1.0, 0.0)], [f"flood label ({int(y[land_mask].sum())} cells)"],
        f"7. Output label y - {ld}", cmap="Reds", cols=1)

## 12. Materialize the cache

Precompute every 2019-2026 sample to CACHE_DIR in parallel. Per sample we write four files:
the per-frame sequence grid (seq), the per-cell daily summaries (sum), the lead time (t),
and the label (y). All small (~3.6 MB/sample) - the cache is ~9 GB total. Reads come from the GOES HDD (disk4), writes
go to the NVMe — different drives, no contention — so ~16 workers saturate the read side.
Set BUILD_CACHE = True to build all of 2019-2026 (**~9 GB**; the build takes a few hours,
HDD-read-bound); leave it False for a 2-sample smoke test. A progress line prints every
PROGRESS_EVERY completed samples (count, %, GB written, samples/min, elapsed, ETA).

The writer skips any day whose _sum.npy already exists, so the build is resumable — rerun to
pick up where it stopped. (If you ever change the feature/resolution contract, delete
CACHE_DIR first so stale files are not mixed in.)

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

CACHE_DIR.mkdir(parents=True, exist_ok=True)
index.to_parquet(CACHE_DIR / "manifest.parquet")
print(f"wrote manifest ({len(index)} samples) -> {CACHE_DIR}")

BUILD_CACHE = True           # <- True to build ALL 2019-2026 samples (~9 GB per-cell)
N_WORKERS = 16
PROGRESS_EVERY = 25                  # print a progress line every N completed samples


def _write_one(args):
    "Build+save one sample. Returns (day, nbytes, err): err=None ok, nbytes=0 cached."
    in_day, label_day = args
    stem = CACHE_DIR / f"{label_day:%Y%m%d}"
    day = f"{label_day:%Y%m%d}"
    if Path(f"{stem}_sum.npy").exists():
        return (day, 0, None)                             # already cached (resumable)
    for attempt in range(2):                              # one retry (transient HDD/HDF reads)
        try:
            s = build_sample(in_day, label_day)
            np.save(f"{stem}_seq.npy", s["seq"])
            np.save(f"{stem}_t.npy", s["t"])
            np.save(f"{stem}_y.npy", s["y"])
            np.save(f"{stem}_sum.npy", s["sum"])          # sum LAST: its presence = complete
            return (day, int(sum(v.nbytes for v in s.values())), None)
        except Exception as e:                            # corrupt/missing GOES file etc.
            if attempt == 0:
                continue
            return (day, 0, f"{type(e).__name__}: {e}")


todo = list(zip(index["in_day"].dt.date, index["label_day"].dt.date))
if not BUILD_CACHE:
    todo = todo[:2]
    print(f"smoke test: writing {len(todo)} samples (set BUILD_CACHE=True for all)")
else:
    print(f"building {len(todo)} samples on {N_WORKERS} workers "
          f"(~{len(todo)*sum(v.nbytes for v in _s.values())/1e9:.1f} GB, "
          f"HDD-read-bound) ...", flush=True)

t0 = time.perf_counter()
n_total, done, built, gb, failed = len(todo), 0, 0, 0.0, []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_write_one, a) for a in todo]
    for fut in as_completed(futures):
        day, nbytes, err = fut.result()                   # never raises (caught in worker)
        done += 1
        if err:
            failed.append((day, err))
        else:
            built += nbytes > 0
            gb += nbytes / 1e9
        if done % PROGRESS_EVERY == 0 or done == n_total:
            el = time.perf_counter() - t0
            rate = done / el                                  # samples/sec
            eta = (n_total - done) / rate / 60 if rate else 0  # minutes
            print(f"  [{done:>4}/{n_total}] {done/n_total:4.0%}  "
                  f"built {built:>4}  failed {len(failed):>2}  {gb:6.1f} GB  "
                  f"{rate * 60:5.1f}/min  elapsed {el / 60:5.1f}m  eta {eta:5.1f}m",
                  flush=True)
print(f"done: {built} written, {done - built - len(failed)} cached, {len(failed)} failed "
      f"in {(time.perf_counter() - t0) / 60:.1f} min -> {CACHE_DIR}")
if failed:
    print(f"\n{len(failed)} samples failed (corrupt/missing GOES files). Re-download those "
          f"GOES days and rerun this cell (it resumes):")
    for day, err in failed[:25]:
        print(f"  {day}: {err}")
    if len(failed) > 25:
        print(f"  ... and {len(failed) - 25} more")

## 13. Summary — features and dataset

A full accounting of what the cache holds: every feature group with its shape and channel
names, then the dataset totals — date range, sample counts and split, and the flooded vs
non-flooded land-cell balance (the class-imbalance the model faces).

In [ ]:
man = pd.read_parquet(CACHE_DIR / "manifest.parquet")
y_files = sorted(CACHE_DIR.glob("*_y.npy"))

print("=" * 78)
print(f"UNIFIED CACHE  ({min(YEARS)}-{max(YEARS)})  ->  {CACHE_DIR}")
print("=" * 78)

groups = [
    ("seq (per-cell, per frame)", f"(8, {N_SEQ}, {GRID_R}, {GRID_C})", "f32", SEQ_NAMES),
    ("sum (per-cell daily summary)", f"({N_SUM}, {GRID_R}, {GRID_C})", "f32", SUM_NAMES),
    ("t   (lead time)", "(8,)", "f32", ["hours_before_D / 24"]),
    ("y   (label)", f"({GRID_R}, {GRID_C})", "u8", ["1 = observed flood (gs U storm)"]),
]
print("\nFEATURES")
for name, shape, dt, feats in groups:
    print(f"\n  {name}  shape {shape}  {dt}")
    for i in range(0, len(feats), 3):
        print("      " + ", ".join(feats[i:i + 3]))

print("\n" + "=" * 78)
print("DATASET")
print("=" * 78)
print(f"  date range (label day) : {man['label_day'].min().date()} -> {man['label_day'].max().date()}")
print(f"  total samples (days)   : {len(man)}")
sp = man['split'].value_counts().reindex(['cv', 'unused']).fillna(0).astype(int)
print(f"  CV pool (2019-2025)    : {sp['cv']} samples    unused (2026): {sp['unused']}")
print(f"  split                  : blocked K-fold CV at train time (foldsplit.py)")
print(f"  cached on disk so far  : {len(y_files)} samples")

if y_files:
    man = man.set_index(man["label_day"].dt.strftime("%Y%m%d"))
    cv_days = [d for d in man.index[man["split"] == "cv"]
               if (CACHE_DIR / f"{d}_y.npy").exists()]
    yr_of = {d: int(d[:4]) for d in cv_days}
    print(f"\n  {'year':<7}{'days':>6}{'mean pos/day':>14}{'max':>6}{'base rate':>12}")
    for y in sorted(set(yr_of.values())):
        days = [d for d in cv_days if yr_of[d] == y]
        Y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days]).astype(bool)
        ld = land_mask[None]
        per_day = (Y & ld).sum((1, 2))
        base = (Y & ld).sum() / (land_mask.sum() * len(Y))
        print(f"  {y:<7}{len(days):>6}{per_day.mean():>14.1f}{int(per_day.max()):>6}{base:>11.3%}")
    print(f"\n  land cells per day: {n_land}  |  positives are rare -> recall-favoring loss")
else:
    print("\n  (no samples cached yet - set BUILD_CACHE=True and run the materialize cell)")